# inference-mode-step — ex3: step body wrapped in `with t.no_grad():` (no decorator)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inference-mode-step`. Running the final beacon cell reports progress against the `PyTorch: Inference mode step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Inference mode step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inference-mode-step`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inference-mode-step"
DD_SUBTOPIC = "PyTorch: Inference mode step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## inference_mode for optimizer.step — quick refresher

The `theta -= lr * grad` in-place update on a leaf tensor with `requires_grad=True` raises:
```
RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.
```
There are TWO standard ways to silence this safely:
1. Decorate the step method: `@t.inference_mode()`.
2. Wrap the body in a context manager: `with t.no_grad():` (or `with t.inference_mode():`).

**This drill (ex3) vs prior.** ex1 demonstrated the decorator form. ex2 diagnosed the missing-decorator error. ex3 uses the **context-manager** form — functionally equivalent but the decision point is granularity: a context manager scopes to specific lines (e.g. you want autograd ON for some pre-step logging but OFF for the actual update).

### Exercise 3 — step body wrapped in `with t.no_grad():` (no decorator)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `with t.no_grad():` inside an optimizer's `step` to achieve the same leaf-in-place permission as `@t.inference_mode()`, and verify the parameter trajectory is identical to the decorator form.
> Keywords: no_grad, context-manager, inference-mode, scoped-disable
> ```

**KCs targeted:** `inference-mode-allows-leaf-in-place-mutation`, `no-grad-context-manager-equivalence`

Implement `Ex3ContextSGD` — a minimal SGD optimizer whose `step` uses a `with t.no_grad():` block instead of the `@t.inference_mode()` decorator.

1. `__init__(self, params, lr)`: materialize `self.params = list(params)`, store `self.lr = lr`.
2. `step(self)`:
   - NO decorator.
   - Wrap the entire body in `with t.no_grad():`.
   - Inside the block, for each `p` with non-None `.grad`, do the BARE in-place update `p -= self.lr * p.grad` (NOT `p.data -= ...`).
3. `zero_grad(self)`: set every `p.grad = None`.

**What the test verifies.** It runs your optimizer side-by-side with a reference decorator-style optimizer for 5 steps on the same model + same loss, and asserts the parameter trajectories are IDENTICAL (allclose with atol=1e-6). The point: `with t.no_grad()` and `@t.inference_mode()` produce the same behavior for this in-place update pattern.

**No decorator allowed.** The test also inspects `Ex3ContextSGD.step` for the absence of `__wrapped__` (which decorators set). A decorated step would still pass the trajectory check but fails the no-decorator structural check — the LO is specifically about the context-manager form.

In [ ]:
class Ex3ContextSGD:
    def __init__(self, params, lr):
        raise NotImplementedError()
    def step(self):
        raise NotImplementedError()
    def zero_grad(self):
        raise NotImplementedError()


def _test_ex3():
    # === Reference: decorator-style SGD (mirrors ex1) ===
    class _RefDecoratorSGD:
        def __init__(self, params, lr):
            self.params = list(params)
            self.lr = lr
        @t.inference_mode()
        def step(self):
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad
        def zero_grad(self):
            for p in self.params:
                p.grad = None

    # === Build TWO models with identical init, run 5 steps each ===
    def _build_model_and_data(seed=0):
        g = t.Generator().manual_seed(seed)
        w = t.nn.Parameter(t.randn(3, 2, generator=g))
        b = t.nn.Parameter(t.zeros(2))
        X  = t.randn(8, 3, generator=g)
        Y  = t.randn(8, 2, generator=g)
        return [w, b], X, Y

    def _train_5_steps(opt_cls, lr=0.05):
        params, X, Y = _build_model_and_data(seed=42)
        opt = opt_cls(params, lr=lr)
        history = []
        for _ in range(5):
            opt.zero_grad()
            pred = X @ params[0] + params[1]
            loss = ((pred - Y) ** 2).mean()
            loss.backward()
            opt.step()
            history.append((params[0].detach().clone(), params[1].detach().clone()))
        return history

    ref_traj  = _train_5_steps(_RefDecoratorSGD)
    ours_traj = _train_5_steps(Ex3ContextSGD)

    assert len(ref_traj) == len(ours_traj) == 5
    for i, ((rw, rb), (ow, ob)) in enumerate(zip(ref_traj, ours_traj)):
        assert t.allclose(rw, ow, atol=1e-6), f'step {i}: weight mismatch\n  ref={rw}\n  ours={ow}'
        assert t.allclose(rb, ob, atol=1e-6), f'step {i}: bias mismatch'

    # === Structural check: step uses context manager, NOT decorator ===
    import inspect
    step_src = inspect.getsource(Ex3ContextSGD.step)
    assert 'with t.no_grad' in step_src or 'with torch.no_grad' in step_src, (
        f'step body must contain a `with t.no_grad():` block. Got:\n{step_src}'
    )
    assert '@t.inference_mode' not in step_src and '@torch.inference_mode' not in step_src, (
        f'step must NOT use the @inference_mode decorator (drill scope is context-manager). Got:\n{step_src}'
    )

    # === Sanity: step doesn't raise on a fresh leaf param ===
    p = t.nn.Parameter(t.randn(4))
    p.grad = t.ones(4)
    opt = Ex3ContextSGD([p], lr=0.1)
    before = p.detach().clone()
    opt.step()  # must not raise the leaf-in-place RuntimeError
    after = p.detach().clone()
    assert t.allclose(after, before - 0.1), f'one step should move by -lr*grad: {after} vs {before - 0.1}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
class Ex3ContextSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
    def step(self):
        with t.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad
    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Decorator vs context manager — same effect, different scope.** `@t.inference_mode()` (or `@t.no_grad()`) disables autograd for the ENTIRE function body. `with t.no_grad():` disables it only for the indented block. For an optimizer.step() that's a distinction without a difference — but if you wanted to log `loss.item()` AFTER the update (with grad re-enabled for a later backward), the context manager gives you that control.

**`t.no_grad()` vs `t.inference_mode()`.** Both disable autograd. `inference_mode()` is slightly faster (skips version-counter bookkeeping) and disallows view-tracking; `no_grad` is the older, more permissive form. For optimizer steps either works — most codebases use `no_grad` out of habit, ARENA's reference solution uses `inference_mode`.

**The trajectory test.** Identical seeds + identical learning rate + identical loss + same in-place update mechanic ⇒ bit-identical parameter trajectories. Any divergence would indicate the two modes aren't actually equivalent for this pattern (they are).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()